# Parquet Basics — 05: Column Pruning

## What you will learn
Column pruning is Parquet's second superpower after predicate pushdown.
When you select only a few columns, Spark reads ONLY those column bytes
from disk — skipping all other columns entirely.

In this notebook:
1. How column pruning works in the Parquet format
2. Measuring the impact: SELECT 1 column vs SELECT all
3. Nested column pruning — structs and maps
4. Pitfalls that defeat column pruning
5. `select()` vs `drop()` — which preserves pruning
6. Column pruning with explode() — the hidden cost


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty  = random.randint(1, 20)
    price = round(random.uniform(5, 2000), 2)
    disc  = round(random.uniform(0, 0.4), 3)
    rev   = round(qty * price * (1 - disc), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,50000),
                 f"Product_{random.randint(1,500)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, disc, rev,
                 random.choice(STATUSES), random.random() < 0.05, d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows | {len(schema)} columns")
df.printSchema()

In [ ]:
PQ_PATH = f"{DATA_DIR}/column_pruning_test"
df.write.mode("overwrite").parquet(PQ_PATH)

import glob as glib
size_mb = sum(pathlib.Path(f).stat().st_size
              for f in glib.glob(f"{PQ_PATH}/*.parquet")) / 1024/1024
print(f"Written {N:,} rows, {len(df.columns)} columns, {size_mb:.1f} MB total")

## Step 1 — Column Pruning in Action

Parquet stores each column in separate byte ranges within each row group.
Reading only the `revenue` column means Spark reads only that column's bytes
— completely skipping bytes for all other 11 columns.


In [ ]:
# SELECT all columns
t_all = []
for _ in range(3):
    t0 = time.time()
    spark.read.parquet(PQ_PATH).agg(F.sum("revenue")).collect()
    t_all.append(time.time() - t0)

# SELECT only 1 column (revenue)
t_one = []
for _ in range(3):
    t0 = time.time()
    spark.read.parquet(PQ_PATH).select("revenue").agg(F.sum("revenue")).collect()
    t_one.append(time.time() - t0)

# SELECT 2 columns
t_two = []
for _ in range(3):
    t0 = time.time()
    spark.read.parquet(PQ_PATH).select("revenue","category").agg(F.sum("revenue")).collect()
    t_two.append(time.time() - t0)

print("Column pruning benchmark:")
print(f"  SELECT *         (12 cols): {sorted(t_all)[1]:.3f}s  (reads all bytes)")
print(f"  SELECT revenue    (1 col) : {sorted(t_one)[1]:.3f}s  (reads ~8% of bytes)")
print(f"  SELECT 2 columns  (2 cols): {sorted(t_two)[1]:.3f}s  (reads ~16% of bytes)")
print()
print("Verify in explain — look for 'ReadSchema' in FileScan node:")
spark.read.parquet(PQ_PATH).select("revenue","category") \
     .explain(mode="formatted")

## Step 2 — Nested Column Pruning: Structs

When your Parquet has struct columns, you can prune at the field level.


In [ ]:
# Create DataFrame with struct column
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

nested_schema = StructType([
    StructField("order_id", LongType()),
    StructField("customer", StructType([
        StructField("id",      LongType()),
        StructField("name",    StringType()),
        StructField("country", StringType()),
        StructField("email",   StringType()),    # large string field
    ])),
    StructField("amount",    DoubleType()),
    StructField("status",    StringType()),
])

nested_data = spark.createDataFrame([
    (i, (i, f"Customer_{i}", "US", f"cust{i}@example.com"), float(i*100), "confirmed")
    for i in range(100_000)
], nested_schema)

nested_path = f"{DATA_DIR}/nested_structs"
nested_data.write.mode("overwrite").parquet(nested_path)
print("Written nested struct Parquet")

# Read only the country field inside customer struct
t0 = time.time()
spark.read.parquet(nested_path) \
     .select("order_id", "customer.country", "amount") \
     .agg(F.count("*")).collect()
t_pruned = time.time() - t0

# Read entire customer struct (includes large email field)
t0 = time.time()
spark.read.parquet(nested_path) \
     .select("order_id", "customer", "amount") \
     .agg(F.count("*")).collect()
t_full = time.time() - t0

print(f"\nNested column pruning:")
print(f"  Select customer.country only : {t_pruned:.3f}s  (skips email, name fields)")
print(f"  Select customer struct (all) : {t_full:.3f}s  (reads all nested fields)")

print("\nExplain — note ReadSchema includes only needed nested fields:")
spark.read.parquet(nested_path).select("order_id","customer.country").explain(mode="formatted")

## Step 3 — Pitfalls That Defeat Column Pruning

These patterns look innocent but force Spark to read all columns.


In [ ]:
pq_df = spark.read.parquet(PQ_PATH)

# Pitfall 1: Using * in select after withColumn
print("PITFALL 1: select('*') — reads ALL columns")
t0 = time.time()
pq_df.select("*").agg(F.sum("revenue")).collect()
t_star = time.time() - t0
print(f"  select('*') time: {t_star:.3f}s  (no pruning!)")

# Pitfall 2: toDF() with all columns
print("\nPITFALL 2: calling toDF() expands to all columns")

# Pitfall 3: cache() before select — caches all columns then selects
print("\nPITFALL 3: cache BEFORE select defeats pruning")
t0 = time.time()
pq_df.cache().select("revenue").agg(F.sum("revenue")).collect()
t_cache_before = time.time() - t0
print(f"  cache() then select: {t_cache_before:.3f}s  (cached ALL cols, then filtered)")

pq_df.unpersist()

# Correct: select BEFORE cache
t0 = time.time()
pq_df.select("revenue").cache().agg(F.sum("revenue")).collect()
t_select_first = time.time() - t0
print(f"  select() then cache: {t_select_first:.3f}s  (pruned BEFORE caching)")

pq_df.select("revenue").unpersist()
print()
print("Rule: select() BEFORE cache() to prune columns early")

## Step 4 — select() vs drop() for Pruning


In [ ]:
# select() — explicit about what you need (Spark can prune)
t0 = time.time()
spark.read.parquet(PQ_PATH) \
     .select("order_id","revenue","category") \
     .agg(F.sum("revenue")).collect()
t_select = time.time() - t0

# drop() — removes columns but Spark must still read them all first
t0 = time.time()
cols_to_drop = [c for c in df.columns if c not in ["order_id","revenue","category"]]
spark.read.parquet(PQ_PATH) \
     .drop(*cols_to_drop) \
     .agg(F.sum("revenue")).collect()
t_drop = time.time() - t0

print("select() vs drop() for column reduction:")
print(f"  select(3 cols)     : {t_select:.3f}s  (Spark prunes at read — reads only 3 cols)")
print(f"  drop({len(cols_to_drop)} cols) : {t_drop:.3f}s  (Spark reads all cols, then drops)")
print()
print("ALWAYS use select() when you know which columns you need.")
print("Catalyst may optimize drop() in some cases, but select() is explicit.")

## Summary: Column Pruning Rules

### Always use `select()` to specify needed columns
```python
# Good — only revenue and category bytes read from disk
df = spark.read.parquet(path).select("revenue", "category")

# Bad — all columns read, then most discarded
df = spark.read.parquet(path).drop("product", "customer_id", ...)
```

### Nested columns
```python
# Read only one field from a struct
df.select("order_id", "customer.country")   # skips customer.name, customer.email
```

### Cache after select, not before
```python
# Good — only 2 columns cached
spark.read.parquet(path).select("revenue","category").cache()

# Bad — all 12 columns cached, wasting memory
spark.read.parquet(path).cache().select("revenue","category")
```

### Column pruning effectiveness by workload
| Query pattern | Pruning benefit |
|---|---|
| Aggregation on 1-2 cols | Very high (90%+ I/O reduction) |
| JOIN on key + few cols | High |
| SELECT * | Zero benefit |
| Wide row reads | Low |
